In [2]:
# data ingestion 
from langchain_community.document_loaders import TextLoader

loader = TextLoader("speech.txt")
text_docs = loader.load()
text_docs

[Document(metadata={'source': 'speech.txt'}, page_content='A well-written speech is a memorable one, and when tasked with giving a speech, \nthis is one of your primary goals. You may also have a secondary goal,\nlike teaching the audience something new, congratulating one or more people,\npersuading listeners to take a specific position, or promoting yourself or another individual.\n\n')]

In [3]:
import os 
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [4]:
# web base loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

# load, chunk and index the content of a html page
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                    bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                        class_=("post-title", "post-content", "post-header")
                    )))

text_docs = loader.load()


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
text_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistake

In [6]:
# Pdf reader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("The Power of Habit - Charles Duhigg.pdf")
pdf_docs = loader.load()
pdf_docs

[Document(metadata={'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2012-03-12T03:38:29+02:00', 'title': 'The Power of Habit: Why We Do What We Do in Life and Business', 'author': 'Charles Duhigg', 'source': 'The Power of Habit - Charles Duhigg.pdf', 'total_pages': 156, 'page': 0, 'page_label': '1'}, page_content='The Pow er of Habit  is a work of nonfiction. Nonetheless, some names and personal characteristics of individuals or events have been changed in order\nto disguise identities. Any resulting resemblance to persons living or dead is entirely coincidental and unintentional.\nCopyright © 2012 by Charles Duhigg\nAll rights reserved.\nPublished in the United States by Random House, an imprint of \nThe Random House Publishing Group, a division of Random House, Inc., New York.\nR ANDOM  H OUSE  and colophon are registered trademarks of Random House, Inc.\nLibrary of Congress Cataloging-in-Publication Data\nDuhigg, Char

In [7]:
# pdf convert to chunks 
# chunking the text
# we will use RecursiveCharacterTextSplitter to chunk the text into smaller pieces
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    )

pdf_docs = text_splitter.split_documents(pdf_docs)

In [8]:
pdf_docs[:5]

[Document(metadata={'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2012-03-12T03:38:29+02:00', 'title': 'The Power of Habit: Why We Do What We Do in Life and Business', 'author': 'Charles Duhigg', 'source': 'The Power of Habit - Charles Duhigg.pdf', 'total_pages': 156, 'page': 0, 'page_label': '1'}, page_content='The Pow er of Habit  is a work of nonfiction. Nonetheless, some names and personal characteristics of individuals or events have been changed in order\nto disguise identities. Any resulting resemblance to persons living or dead is entirely coincidental and unintentional.\nCopyright © 2012 by Charles Duhigg\nAll rights reserved.\nPublished in the United States by Random House, an imprint of \nThe Random House Publishing Group, a division of Random House, Inc., New York.\nR ANDOM  H OUSE  and colophon are registered trademarks of Random House, Inc.\nLibrary of Congress Cataloging-in-Publication Data\nDuhigg, Char

In [9]:
# now covert to Vector
# vector Embedding and vectore store 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Create Gemini embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
#  Create a Chroma vector store from the documents
# Note: You can use the first 15 documents for testing purposes
db = Chroma.from_documents(documents=pdf_docs[:15], embedding=embedding_model)


In [10]:
# Vector Database
query = "What is the main idea of the book 'The Power of Habit'?"
# result = db.similarity_search(query, k=3)

# Try with different search types
result = db.similarity_search_with_score(query, k=3)  # Get scores too
print(result)

[(Document(metadata={'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'page_label': '1', 'moddate': '2012-03-12T03:38:29+02:00', 'creator': 'PyPDF', 'source': 'The Power of Habit - Charles Duhigg.pdf', 'total_pages': 156, 'title': 'The Power of Habit: Why We Do What We Do in Life and Business', 'author': 'Charles Duhigg', 'creationdate': '', 'page': 0}, page_content='eISBN: 978-0-679-60385-6\n1. \u2002 Habit.   2. \u2002 Habit—Social aspects.   3. \u2002 Change (Psychology)   I. \u2002 Title.\nBF335.D76 2012\n158.1—dc23                      2011029545\nIllustration on this page\n by Andrew PoleAll other illustrations by Anton Ioukhnovets\nwww.atrandom.com\nv3.1'), 0.48229557275772095), (Document(metadata={'title': 'The Power of Habit: Why We Do What We Do in Life and Business', 'page': 1, 'creator': 'PyPDF', 'total_pages': 156, 'author': 'Charles Duhigg', 'page_label': '2', 'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'source': 'The Power of Habit - Charle

In [11]:
# result[0].page_content

In [12]:
# Faiss Vector Database
from langchain_community.vectorstores import FAISS

# Create FAISS vector store from documents
faiss_db = FAISS.from_documents(
    documents=pdf_docs[:15],  # Using first 15 documents (adjust as needed)
    embedding=embedding_model
)


In [13]:
# Save the FAISS index locally (optional)
faiss_db.save_local("faiss_index")  # Saves to disk for later reuse

In [14]:
# Example query
query1 = "What is the main idea of the book 'The Power of Habit'?"
results = faiss_db.similarity_search(query1, k=3)  # Get top 3 most similar documents

In [15]:
# Display results
for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:200] + "...")  # Print first 200 chars of each result


Result 1:
eISBN: 978-0-679-60385-6
1.   Habit.   2.   Habit—Social aspects.   3.   Change (Psychology)   I.   Title.
BF335.D76 2012
158.1—dc23                      2011029545
Illustration on this page
 by Andre...

Result 2:
CONTENTS
Cover
Title Page
Copyright
Dedication
PROLOGUE
The Habit Cure
PART ONE
The Habits of Individuals
1. THE HABIT LOOP
How Habits Work
2. THE CRAVING BRAIN
How to Create New Habits
3. THE GOLDEN ...

Result 3:
The Pow er of Habit  is a work of nonfiction. Nonetheless, some names and personal characteristics of individuals or events have been changed in order
to disguise identities. Any resulting resemblance...


In [16]:
# Design chatprompt template 
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.
    Answer the question based on the context provided below.
    I will tip you $100 if user is satified with your answer.
    <context>
    {context}
    </context>
    Question : {input}"""
)

In [19]:
# from langchain_community.llms import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(
    model = "gemini-2.0-flash",  # Use the appropriate model name
    temperature=0.6,  # Adjust temperature for response variability
)

In [20]:
# chain introduction
# create a stuff DocumentChain
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(
    llm, 
    prompt
)

In [21]:
"""
#Retrievers: A retriever is a component that retrieves relevant documents from a vector store based on a query.
Retrievers can be used to fetch documents that are relevant to a user's query, which can then be processed by a chain to generate a response.
from langchain.chains import RetrievalQA
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":
3})  # Adjust k for number of documents to retrieve
retrieval_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True  # Set to True to return source documents
)
"""

retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000022C4E64F8B0>, search_kwargs={})

In [23]:
"""
Retrieval chain: A retrieval chain combines a retriever with a document processing chain to answer questions based on retrieved documents.
This chain first retrieves relevant documents using the retriever and then processes those documents using the document chain
"""
from langchain.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(
    retriever, 
    document_chain
)

In [36]:
response = retrieval_chain.invoke({"input" : "who is Pole, and was it a statistician.yes or not?"})

In [37]:
response["answer"]

'Based on the context, Andrew Pole is an illustrator. The text does not mention anything about him being a statistician.\nAnswer: no'